# 比特频谱校准

通过当前设备的 Active 配置运行单比特或双比特并行频谱。实验结果自动写入 `output/experiments/`，Web 控制台只负责查看结果。

In [1]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise RuntimeError('未找到 SQC_simulation 项目根目录')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from sqvm.calibration import (
    SpectroscopyAxis,
    SpectroscopyCalibrationPolicy,
    SpectroscopyCalibrationRequest,
    SpectroscopyMode,
    SpectroscopyPulsePolicy,
    SpectroscopyRequest,
    run_active_qubit_spectroscopy_calibration,
)

print(f'项目目录: {PROJECT_ROOT}')

项目目录: D:\Codex\SQC_simulation


## 扫描参数

`TARGETS=("Q1",)` 表示单比特扫描；`TARGETS=("Q1", "Q2")` 表示两个比特按相同点数并行扫描。

In [2]:
DEVICE_ID = 'demo_2q1c2r'
TARGETS = ('Q1', 'Q2')
FREQUENCIES_GHZ = {
    'Q1': (4.90, 5.00, 5.10),
    'Q2': (5.10, 5.20, 5.30),
}
PULSE_LENGTH_SAMPLES = 12
PULSE_AMPLITUDE_GHZ = 0.02
R_SIGMA_SAMPLES = 2.5
TIMEOUT_S = 1800.0
RUN_EXPERIMENT = False

In [5]:
mode = SpectroscopyMode.SINGLE if len(TARGETS) == 1 else SpectroscopyMode.PARALLEL_LOCKSTEP
coarse_request = SpectroscopyRequest(
    execution_mode=mode,
    run_phase='coarse',
    targets=TARGETS,
    axes=tuple(SpectroscopyAxis(target, FREQUENCIES_GHZ[target]) for target in TARGETS),
    pulse_policies=tuple(
        SpectroscopyPulsePolicy(
            qagent=target,
            length_samples=PULSE_LENGTH_SAMPLES,
            amplitude_GHz=PULSE_AMPLITUDE_GHZ,
            r_sigma_samples=R_SIGMA_SAMPLES,
        )
        for target in TARGETS
    ),
)
request = SpectroscopyCalibrationRequest(
    coarse_request=coarse_request,
    policy=SpectroscopyCalibrationPolicy(
        fine_span_GHz=0.10,
        fine_points=15,
        confirmation_span_GHz=0.08,
        confirmation_points=5,
        min_contrast=0.01,
    ),
)
request

SpectroscopyCalibrationRequest(coarse_request=SpectroscopyRequest(execution_mode=<SpectroscopyMode.PARALLEL_LOCKSTEP: 'parallel_lockstep'>, run_phase='coarse', targets=('Q1', 'Q2'), axes=(SpectroscopyAxis(qagent='Q1', frequencies_GHz=(4.9, 5.0, 5.1)), SpectroscopyAxis(qagent='Q2', frequencies_GHz=(5.1, 5.2, 5.3))), pulse_policies=(SpectroscopyPulsePolicy(qagent='Q1', length_samples=12, amplitude_GHz=0.02, r_sigma_samples=2.5, policy_id='qubit_spectroscopy_pulse_policy_v1'), SpectroscopyPulsePolicy(qagent='Q2', length_samples=12, amplitude_GHz=0.02, r_sigma_samples=2.5, policy_id='qubit_spectroscopy_pulse_policy_v1')), max_points=64), policy=SpectroscopyCalibrationPolicy(fine_span_GHz=0.1, fine_points=15, confirmation_span_GHz=0.08, confirmation_points=5, min_contrast=0.01, near_peak_tolerance=1e-12, max_leakage=0.05, max_norm_error=1e-08, max_coarse_refined_shift_GHz=0.1, max_parallel_peak_shift_GHz=0.02, max_cross_excitation=0.05, max_parallel_leakage_delta=0.02))

In [6]:
if RUN_EXPERIMENT:
    result = run_active_qubit_spectroscopy_calibration(
        request,
        device_id=DEVICE_ID,
        repository_root=PROJECT_ROOT,
        timeout_s=TIMEOUT_S,
    )
    print(f'运行 ID: {result.run_id}')
    print(f'结果目录: {result.root}')
    print(f'候选可用: {result.recommendation_eligible}')
    print(f'候选参数: {dict(result.candidates)}')
else:
    print('参数已构造。将 RUN_EXPERIMENT 改为 True 后重新运行本单元格。')

参数已构造。将 RUN_EXPERIMENT 改为 True 后重新运行本单元格。


## 查看结果

启动 `start_calibration_web.cmd` 后访问 [http://127.0.0.1:8765/#/experiments](http://127.0.0.1:8765/#/experiments)。